# Multi-Query Expansion
> Generate multiple query variations to improve retrieval recall.

`MultiQueryTransformer` produces several rephrased versions of the
original query. Each variation is used for retrieval independently,
and results are merged. This captures different facets of the user's
intent.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.query import MultiQueryTransformer
from anchor.models import QueryBundle

## Define a Multi-Query Generator
The generator takes a query string and the number of variations to
produce. In production this would be an LLM call.

In [ ]:
def mock_generate(query: str, num: int) -> list:
    """Generate num query variations from the original query."""
    variations = [
        f"{query} - explained simply",
        f"What are the key concepts of {query.lower()}?",
        f"Practical guide to {query.lower()}",
        f"{query} best practices and patterns",
        f"How does {query.lower()} work in practice?",
    ]
    return variations[:num]

# Test the generator
results = mock_generate("context engineering", 3)
print(f"Generated {len(results)} variations:")
for i, v in enumerate(results):
    print(f"  [{i}] {v}")

## Create the Multi-Query Transformer
Set `num_queries` to control how many variations are generated per query.

In [ ]:
multi = MultiQueryTransformer(generate_fn=mock_generate, num_queries=3)

print(f"Transformer: {type(multi).__name__}")
print(f"Num queries: {multi.num_queries}")

## Transform a Query
`transform()` returns a list of `QueryBundle` objects, one for each
generated variation.

In [ ]:
query = QueryBundle(query_str="What is context engineering?")
print(f"Original: {query.query_str}\n")

expanded = multi.transform(query)

print(f"Expanded to {len(expanded)} queries:")
for i, q in enumerate(expanded):
    print(f"  [{i}] {q.query_str}")

## Vary the Number of Queries
More variations improve recall but increase retrieval cost.

In [ ]:
for n in [2, 3, 5]:
    transformer = MultiQueryTransformer(generate_fn=mock_generate, num_queries=n)
    results = transformer.transform(query)
    print(f"num_queries={n}: {len(results)} queries generated")
    for q in results:
        print(f"    {q.query_str}")
    print()

## Multi-Query Retrieval Pattern
In a full pipeline, each variation retrieves independently and results
are merged with deduplication.

In [ ]:
# Simulate multi-query retrieval
def mock_retrieve(query_str: str) -> list:
    """Return mock document IDs based on query content."""
    base = hash(query_str) % 100
    return [f"doc-{base + i}" for i in range(3)]

expanded = multi.transform(query)

all_results = []
for q in expanded:
    docs = mock_retrieve(q.query_str)
    all_results.extend(docs)
    print(f"  Query: {q.query_str[:50]}...")
    print(f"    Retrieved: {docs}")

# Deduplicate
unique_docs = list(dict.fromkeys(all_results))
print(f"\nTotal retrieved: {len(all_results)}")
print(f"After dedup:     {len(unique_docs)}")
print(f"Unique docs:     {unique_docs}")

## Key Takeaways
- `MultiQueryTransformer` generates multiple phrasings of a single query
- Each variation retrieves independently, improving recall
- Results are merged and deduplicated across all variations
- `num_queries` controls the recall-vs-cost trade-off
- The generation function can be swapped between mock and real LLM calls

**Next:** [Query Decomposition](03_decomposition.ipynb)